# Phát hiện biển số - A1 (YOLO26n vs YOLOv8n) - Kaggle
Thin driver (không phụ thuộc OS). Toàn bộ logic nằm trong `plate_detect`; notebook chỉ gọi CLI.

**Chỉ chạy trên Kaggle.** Setup clone repo **public**, cài CLI, và trỏ raw A1 tới Kaggle Dataset đã đính kèm. Các cell sau chạy từ gốc repo tại `/kaggle/working`.

**Trước khi chạy:**
1. Trong Notebook settings, bật **Internet: ON** (git clone và pip; yêu cầu tài khoản đã xác minh số điện thoại).
2. Trong Notebook settings, chọn **Accelerator: GPU** (T4x2 hoặc P100).
3. Dùng **Add data**, tìm `duydieunguyen/licenseplates` (tự đính kèm qua kernel-metadata `dataset_sources`).
4. Repo `UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi` phải **public** (clone không dùng token). Nếu private, dùng secret `GH_PAT`; lưu ý Kaggle Secrets không được `kaggle kernels push` mang theo và chỉ hoạt động từ web editor.

In [ ]:
# === Setup (Kaggle) ===
# Clone BRANCH repo PUBLIC vào /kaggle/working, cài CLI, symlink raw A1 tới dataset đã đính kèm.
import os, glob, shutil, socket

assert os.path.isdir("/kaggle"), "this notebook is Kaggle-only"

# --- lỗi sớm, kèm lý do rõ ràng, trước mọi việc chậm ---
# 1) internet
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit(
        "Internet is OFF on this kernel. Settings -> Internet: On "
        "(requires phone verification: kaggle.com/settings). Enable, then re-run."
    )
# 2) GPU có mặt VÀ arch của nó được torch của Kaggle hỗ trợ. torch 2.10+cu128 của Kaggle đã bỏ
#    Pascal (sm_60) -> P100 báo 'CUDA no kernel image'. Cần T4 (sm_75) trở lên.
import torch
print("torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA not available - Settings -> Accelerator: GPU"
cap  = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name(0)
archs = torch.cuda.get_arch_list()
print(f"GPU: {name}  sm_{cap[0]}{cap[1]}  | torch archs: {archs}")
if f"sm_{cap[0]}{cap[1]}" not in archs:
    raise SystemExit(
        f"GPU {name} (sm_{cap[0]}{cap[1]}) NOT supported by torch {torch.__version__} (built for {archs}). "
        "Settings -> Accelerator -> 'GPU T4 x2' (NOT P100), or push with --accelerator NvidiaTeslaT4. Then re-run."
    )
torch.zeros(1, device="cuda") + 1   # bằng chứng một kernel thật sự chạy
print("GPU kernel launch OK")

# 3) dataset đã đính kèm - tự tìm data root (images/train) bất kỳ đâu dưới /kaggle/input
hits = glob.glob("/kaggle/input/**/images/train", recursive=True)
if not hits:
    have = sorted(glob.glob("/kaggle/input/*"))
    raise SystemExit(
        "A1 dataset not attached (no */images/train under /kaggle/input). "
        "Add data -> duydieunguyen/licenseplates, then re-run. "
        f"Currently attached: {have or 'nothing'}"
    )
DATA_ROOT = os.path.abspath(hits[0][: -len("/images/train")])
print("A1 data root:", DATA_ROOT)

REPO   = "/kaggle/working/UIT2026-DoAnCuoiKi"
BRANCH = "feat/plate-detect-a1"                 # package CHƯA merge vào main - clone branch này
RAW    = "data/raw/kaggle_vn_plate_segment"     # layout mà A1Adapter mong đợi: {images,labels}/{train,val}
URL    = "https://github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git"  # repo public, không cần token

# --- clone repo public, feature branch ---
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} --single-branch {URL} {REPO}
assert os.path.isdir(REPO), "clone failed - is the repo public? (make UIT2026-DoAnCuoiKi public, or use a PAT)"
%cd {REPO}
!git rev-parse --abbrev-ref HEAD   # xác nhận đang ở feature branch

# --- cài package -> đưa CLI `plate_detect` vào PATH (giữ nguyên torch của Kaggle) ---
!pip install -q -e src/ml/plate_detection_pipeline

# --- symlink RAW -> dataset root đã tìm thấy ---
if not os.path.isdir(f"{RAW}/images/train"):
    os.makedirs(os.path.dirname(RAW), exist_ok=True)
    if os.path.islink(RAW) or os.path.exists(RAW):
        (os.unlink if os.path.islink(RAW) else shutil.rmtree)(RAW)
    os.symlink(DATA_ROOT, os.path.abspath(RAW))
    print("raw A1 ->", DATA_ROOT)

# kiểm tra layout raw mà adapter đọc (train + val, images + labels)
for s in ("train", "val"):
    for k in ("images", "labels"):
        assert os.path.isdir(f"{RAW}/{k}/{s}"), f"missing {RAW}/{k}/{s} - check split names (val vs valid)"
print("OK - CLI installed, raw A1 ready at", RAW)

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Chuẩn bị dữ liệu: kiểm tra class-map, chia tập, khử trùng lặp (train/test, train/val) và xác thực

In [ ]:
!plate_detect prepare

In [ ]:
!plate_detect check

## 2. Huấn luyện: ma trận đầy đủ @640 (hai model, seed 0,1,2)

In [ ]:
# CHẠY ĐẦY ĐỦ - config mặc định (configs/default.yaml: imgsz 640, epochs 50, patience 10, seeds 0,1,2), cả hai model
!plate_detect train --config src/ml/plate_detection_pipeline/configs/default.yaml --project runs

## 4. Export mô hình tốt nhất sang ONNX (mỗi model và imgsz, kiểm parity)

In [ ]:
# ví dụ; lặp lại cho best run của mỗi model/imgsz:
!plate_detect export --weights runs/yolo26n_s0_640/weights/best.pt --out weights/yolo26n_a1_640.onnx --imgsz 640

## 5. Đánh giá trên tập test A1: bảng so sánh và experiments.csv

In [ ]:
!plate_detect eval --imgszs 640 --project runs --weights-dir weights --sample-image data/processed/a1_det/images/test/$(ls data/processed/a1_det/images/test | head -1)

## 6. Đóng gói kết quả

Nén `runs/`, `weights/`, và `experiments.csv` thành một archive trong `/kaggle/working/`. Kaggle tự lưu `/kaggle/working`; file zip xuất hiện ở tab **Output** và tải được sau session. Không cần Drive mount.

In [ ]:
# === Đóng gói toàn bộ kết quả vào một .zip dưới /kaggle/working ===
import os, datetime

STAMP   = datetime.datetime.now().strftime("%Y%m%d_%H%M")
ARCHIVE = f"/kaggle/working/plate_det_results_{STAMP}.zip"

# gom những gì tồn tại (runs/ = weights+plots+curves, weights/ = ONNX đã export, experiments.csv)
targets = [p for p in ("runs", "weights", "experiments.csv") if os.path.exists(p)]
assert targets, "nothing to zip - run train/export/eval first"
print("zipping:", targets)
!zip -rq "{ARCHIVE}" {" ".join(targets)}
print("archive:", ARCHIVE, f"({os.path.getsize(ARCHIVE)/1e6:.1f} MB)")
print("grab it from the notebook Output tab after the session ends.")